In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm, neighbors, ensemble
from sklearn.metrics import classification_report, accuracy_score
import joblib
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import GridSearchCV

In [ ]:
data_dir = "/home/hpcsap/my pro/deep_and_fake"

real_dir = os.path.join(data_dir, "real_images")
fake_dir = os.path.join(data_dir, "deepfake_images")

real_images = os.listdir(real_dir)
fake_images = os.listdir(fake_dir)

data = []
labels = []

for img in real_images:
    img_path = os.path.join(real_dir, img)
    img_data = cv2.imread(img_path)
    img_data = cv2.resize(img_data, (100, 100))
    data.append(img_data)
    labels.append("Real")

for img in fake_images:
    img_path = os.path.join(fake_dir, img)
    img_data = cv2.imread(img_path)
    img_data = cv2.resize(img_data, (100, 100))
    data.append(img_data)
    labels.append("Deepfake")

data = np.array(data)
labels = np.array(labels)

# Normalize the image data
data = data / 255.0

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(data, labels_encoded, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(100, 100, 3)))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Save the trained model for feature extraction
model.save("cnn_model.h5")

In [ ]:
cnn_model = tf.keras.models.load_model("cnn_model.h5")

train_features = cnn_model.predict(X_train)
test_features = cnn_model.predict(X_test)

In [ ]:
# SVM
svm_model = svm.SVC()
svm_model.fit(train_features, y_train)

# KNN
knn_model = neighbors.KNeighborsClassifier()
knn_model.fit(train_features, y_train)

# Random Forest
rf_model = ensemble.RandomForestClassifier()
rf_model.fit(train_features, y_train)

In [ ]:
# SVM
svm_params = {'C': [1, 10, 100], 'kernel': ['linear', 'rbf']}
svm_tuned = GridSearchCV(estimator=svm_model, param_grid=svm_params, scoring='accuracy', cv=3)
svm_tuned.fit(train_features, y_train)
svm_best = svm_tuned.best_estimator_

svm_pred = svm_best.predict(test_features)
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_classification_report = classification_report(y_test, svm_pred)

# KNN
knn_params = {'n_neighbors': [3, 5, 7]}
knn_tuned = GridSearchCV(estimator=knn_model, param_grid=knn_params, scoring='accuracy', cv=3)
knn_tuned.fit(train_features, y_train)
knn_best = knn_tuned.best_estimator_

knn_pred = knn_best.predict(test_features)
knn_accuracy = accuracy_score(y_test, knn_pred)
knn_classification_report = classification_report(y_test, knn_pred)

# Random Forest
rf_params = {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 20]}
rf_tuned = GridSearchCV(estimator=rf_model, param_grid=rf_params, scoring='accuracy', cv=3)
rf_tuned.fit(train_features, y_train)
rf_best = rf_tuned.best_estimator_

rf_pred = rf_best.predict(test_features)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_classification_report = classification_report(y_test, rf_pred)

print("SVM Accuracy:", svm_accuracy)
print("SVM Classification Report:\n", svm_classification_report)

print("KNN Accuracy:", knn_accuracy)
print("KNN Classification Report:\n", knn_classification_report)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Classification Report:\n", rf_classification_report)

In [ ]:
# Choose the best model based on the highest accuracy
best_model = svm_best if svm_accuracy > knn_accuracy and svm_accuracy > rf_accuracy else knn_best if knn_accuracy > rf_accuracy else rf_best

# Save the best model
joblib.dump(best_model, "best_model.joblib")

In [ ]:
from flask import Flask, render_template, request

app = Flask(__name__)

model = joblib.load("best_model.joblib")

@app.route("/", methods=["GET", "POST"])
def index():
    if request.method == "POST":
        file = request.files["file"]
        if file:
            image = cv2.imdecode(np.frombuffer(file.read(), np.uint8), 1)
            image = cv2.resize(image, (100, 100))
            image = image / 255.0
            image_features = cnn_model.predict(np.array([image]))
            prediction = le.inverse_transform(model.predict(image_features))[0]
            return render_template("index.html", prediction=prediction)
        else:
            return render_template("index.html")
    else:
        return render_template("index.html")


if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)